# RiskPulse: fine-tune FinBERT on finance tweets

Works on **Kaggle** and **Google Colab**.

**Before running on Kaggle:** in the right-hand panel, set **Accelerator → GPU T4 x2** and **Internet → On**
(Kaggle asks you to verify your phone number once to allow both).

**Before running on Colab:** click **Runtime → Change runtime type → T4 GPU → Save**.

Then run each cell in order with **Shift+Enter**, waiting for each to finish.

What this notebook does:
1. Fine-tunes `ProsusAI/finbert` on the **training split** of `zeroshot/twitter-financial-news-sentiment` (9,543 labelled finance tweets).
2. Tests it on the **validation split** (2,388 tweets), which it never sees during training, and compares it with the original FinBERT on exactly the same tweets.
3. Re-scores every tweet and headline in the RiskPulse repo with the fine-tuned model, so you don't have to run it for an hour on your laptop.

In [ ]:
# Step 1: install libraries (about 1 minute)
!pip install -q "transformers>=4.46" datasets accelerate

**Now restart the kernel once**, so Python loads the freshly installed libraries
(skipping this causes errors like `module 'sympy' has no attribute 'printing'`):

- **Kaggle:** click **Run → Restart & clear cell outputs** (or the circular-arrow restart button).
- **Colab:** click **Runtime → Restart session**.

Then continue from **Step 2**. Don't run Step 1 again.

In [ ]:
# Step 2: check the GPU is on (should print True and 'Tesla T4')
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use one GPU, so results are the same on Kaggle and Colab
import torch
print("torch", torch.__version__, "| built with CUDA:", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU in this session. Kaggle: Session options -> Accelerator -> GPU T4 x2 "
                     "(verify your phone first if it is greyed out). Colab: Runtime -> Change runtime type -> T4 GPU. "
                     "The session restarts; then run this notebook again from the top.")
print(torch.cuda.get_device_name(0))

In [ ]:
# Step 3: settings. CHANGE GITHUB_REPO to your repo's link (green Code button on GitHub).
GITHUB_REPO = "https://github.com/avijitdutta1230000-crypto/VIT_Vellore-Avijit_Dutta-hackathon.git"
BASE_MODEL = "ProsusAI/finbert"
HF_MODEL_NAME = "riskpulse-finbert-tfns"   # name of the fine-tuned model on Hugging Face
SEED = 42
MAX_LENGTH = 96

In [ ]:
# Step 4: load the labelled tweets and prepare them
import re
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

URL_RE, SPACE_RE = re.compile(r"https?://\S+"), re.compile(r"\s+")
def clean_text(text):
    # Same cleaning as src/ingestion/common.py in the repo
    return SPACE_RE.sub(" ", URL_RE.sub("", str(text))).strip()

raw = load_dataset("zeroshot/twitter-financial-news-sentiment")
train_all = raw["train"].to_pandas()
test = raw["validation"].to_pandas()     # held out: used ONCE, at the very end

# Dataset labels: 0 Bearish, 1 Bullish, 2 Neutral.
# FinBERT's labels: 0 positive, 1 negative, 2 neutral. Map dataset -> FinBERT ids.
TO_FINBERT = {0: 1, 1: 0, 2: 2}
for df in (train_all, test):
    df["text"] = df["text"].map(clean_text)
    df["label"] = df["label"].map(TO_FINBERT)

# 10% of the training data is kept aside to pick the best epoch,
# so the held-out test set is never used for any decision.
train, dev = train_test_split(train_all, test_size=0.1, stratify=train_all["label"], random_state=SEED)
print(f"train {len(train):,}  dev {len(dev):,}  held-out test {len(test):,}")
print(train["label"].map({0: "positive", 1: "negative", 2: "neutral"}).value_counts())

In [ ]:
# Step 5: helper to measure accuracy and macro-F1
NAMES = {0: "positive", 1: "negative", 2: "neutral"}

def metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    per_class = {}
    for k, name in NAMES.items():
        tp = int(((y_pred == k) & (y_true == k)).sum())
        precision = tp / max(int((y_pred == k).sum()), 1)
        recall = tp / max(int((y_true == k).sum()), 1)
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[name] = {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)}
    return {"accuracy": round(float((y_true == y_pred).mean()), 4),
            "macro_f1": round(sum(c["f1"] for c in per_class.values()) / 3, 4),
            "per_class": per_class}

def confusion(y_true, y_pred):
    t = pd.Series(y_true).map(NAMES).rename("actual")
    p = pd.Series(y_pred).map(NAMES).rename("predicted")
    return pd.crosstab(t, p)

In [ ]:
# Step 6: tokenize, and score the ORIGINAL FinBERT on the held-out test (the "before" number)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

ds = {name: Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True)).map(tokenize, batched=True)
      for name, df in (("train", train), ("dev", dev), ("test", test))}

@torch.no_grad()
def predict(model, texts, batch_size=128):
    model.eval().to("cuda")
    probs = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt").to("cuda")
        probs.append(torch.softmax(model(**enc).logits, dim=-1).float().cpu().numpy())
    return np.concatenate(probs)

base_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL)
base_pred = predict(base_model, test["text"].tolist()).argmax(axis=1)
before = metrics(test["label"], base_pred)
print("Original FinBERT on held-out test:", before["accuracy"], "accuracy,", before["macro_f1"], "macro-F1")
del base_model

In [ ]:
# Step 7: fine-tune (about 3-5 minutes on a T4 GPU)
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments, set_seed

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    m = metrics(labels, logits.argmax(axis=-1))
    return {"accuracy": m["accuracy"], "macro_f1": m["macro_f1"]}

args = TrainingArguments(
    output_dir="finbert-tfns-checkpoints",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    weight_decay=0.01,
    warmup_steps=int(0.1 * 3 * np.ceil(len(train) / 32)),   # warm up over the first 10% of steps
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    fp16=True,
    seed=SEED,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["dev"],
                  processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
                  compute_metrics=compute_metrics)
trainer.train()

In [ ]:
# Step 8: the honest test - fine-tuned model on the held-out tweets it never saw
import json

ft_pred = predict(trainer.model, test["text"].tolist()).argmax(axis=1)
after = metrics(test["label"], ft_pred)

print(pd.DataFrame({"Original FinBERT": {"accuracy": before["accuracy"], "macro_f1": before["macro_f1"]},
                    "Fine-tuned": {"accuracy": after["accuracy"], "macro_f1": after["macro_f1"]}}))
print("\nPer class (fine-tuned):")
print(pd.DataFrame(after["per_class"]).T)
print("\nConfusion matrix (fine-tuned):")
print(confusion(test["label"], ft_pred))

results = {
    "base_model": BASE_MODEL,
    "fine_tuned_on": "zeroshot/twitter-financial-news-sentiment train split (90% train, 10% dev for epoch selection)",
    "tested_on": "zeroshot/twitter-financial-news-sentiment validation split (held out, used once)",
    "n_train": int(len(train)), "n_dev": int(len(dev)), "n_test": int(len(test)),
    "settings": {"learning_rate": 2e-5, "epochs": 3, "batch_size": 32, "max_length": MAX_LENGTH, "seed": SEED},
    "before": before, "after": after,
}
with open("sentiment_eval_finetuned.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved sentiment_eval_finetuned.json")

## Step 9: publish the model on Hugging Face (so the repo can load it)

The model is about 440 MB, too big for GitHub. Hugging Face hosts models for free.

1. Make a free account at **huggingface.co** (if you don't have one).
2. Go to **Settings → Access Tokens → Create new token**, choose type **Write**, and copy it.
3. Run the cell below and paste the token when asked (Kaggle and Colab both show a small login box).

In [ ]:
# Step 9: log in and upload the fine-tuned model
from huggingface_hub import notebook_login, whoami
notebook_login()

In [ ]:
trainer.model.push_to_hub(HF_MODEL_NAME)
tokenizer.push_to_hub(HF_MODEL_NAME)
FULL_MODEL_ID = f"{whoami()['name']}/{HF_MODEL_NAME}"
print("Uploaded. Your model name is:", FULL_MODEL_ID)
print("Put this name in src/nlp/sentiment.py as MODEL_NAME.")

In [ ]:
# Step 10: re-score every tweet and headline in the repo with the fine-tuned model (about 2 minutes)
import os, subprocess, sys
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "-q", GITHUB_REPO, "repo"], check=True)
sys.path.insert(0, "repo")
from src.ingestion.tweets import load_tweets
from src.ingestion.gdelt import load_gdelt

texts = pd.concat([f for f in (load_tweets(), load_gdelt()) if len(f)], ignore_index=True)
texts = texts.drop_duplicates("id")
print(f"Scoring {len(texts):,} texts...")

probs = predict(trainer.model, texts["text"].tolist(), batch_size=256)
id2label = {int(k): v.lower() for k, v in trainer.model.config.id2label.items()}
pos = next(k for k, v in id2label.items() if v == "positive")
neg = next(k for k, v in id2label.items() if v == "negative")
scores = pd.DataFrame({
    "id": texts["id"].values,
    "sentiment": (probs[:, pos] - probs[:, neg]).round(4),
    "sentiment_label": [id2label[i] for i in probs.argmax(axis=1)],
})
scores.to_csv("sentiment.csv", index=False)
print(scores["sentiment_label"].value_counts())
print("Saved sentiment.csv")

In [ ]:
# Step 11: download the two result files to your computer
import os
from IPython.display import FileLink, display
if os.path.exists("/kaggle/working"):
    # Kaggle: click the links below, or use the Output panel on the right (/kaggle/working)
    display(FileLink("sentiment.csv"), FileLink("sentiment_eval_finetuned.json"))
else:
    from google.colab import files   # Colab: downloads start automatically
    files.download("sentiment.csv")
    files.download("sentiment_eval_finetuned.json")